# Clase 8 · Los sonidos del Magdalena (gemelo)

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelve cada uno en la segunda mitad de la clase. Es el gemelo de *El electrocardiograma*: la misma técnica, de la señal a la tabla y de la tabla al modelo, con otra señal.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Dirigir a una IA para que escriba el flujo completo de un clasificador de audio (ventanas, espectrograma, MFCC, cinco modelos, partición por grabación) y auditar lo que escribió. |
| **Enfoque** | Machine Learning. |
| **Resultados de aprendizaje** | 1 (formular), 2 (preparar datos), 3 (flujo reproducible), 4 (implementar y evaluar), 5 (comunicar) y 6 (reflexión ética, técnica y operativa). |
| **Uso de IA** | El código lo escribe Claude, a pedido de ustedes, celda por celda. Ustedes deben poder explicar sin ayuda qué es un espectrograma, qué resume un MFCC, por qué se parte por grabación y no por ventana, y cómo se escogieron las cien grabaciones del experto. |
| **Entregable** | El cuaderno completo y corrido en su portafolio, con las predicciones escritas, la sección de auditoría llena, la decisión en tres frases y la tabla de la variante. |

**Cómo se trabaja.** Las celdas de código están vacías. Cada una trae un comentario que dice qué debe lograr y qué nombres de variable debe dejar. Copien el comentario en Claude, pidan el código, léanlo antes de pegarlo y córranlo. Si falla, lean el error antes de pedir ayuda. Lo único que ya está escrito es el arranque, los parámetros y la carga del dato.

**Cómputo.** Todo corre en CPU. La descarga pesa 257 MB y tarda menos de un minuto. Sacar las características de las mil grabaciones tarda entre uno y dos minutos.

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

PALETA = ["#1D4E89", "#D7263D", "#1B998B", "#6C4AB6", "#5BC0EB", "#F49097", "#9AA5B1"]      # colores del curso
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETA), "axes.titleweight": "bold", "figure.dpi": 110})
print("Herramientas listas.")

In [ ]:
# Los dos parámetros del cuaderno. Al final se cambia la primera línea y se corre todo de nuevo.
SEGUNDOS_OIDOS = 30       # cuántos segundos de cada grabación oye el modelo, contados desde el comienzo
REVISIONES = 100          # cuántas grabaciones alcanza a oír el experto

import time
import librosa
from taller.clase08_senales import grabaciones

## 1. Formulación

Imagínense al equipo que cuida una reserva en el Magdalena Medio. Dejan grabadoras en el bosque y recogen horas de sonido. Saber qué especies cantan, y dónde, es su manera de medir si el bosque se recupera. El equipo tiene un solo ornitólogo capaz de reconocer los cantos de oído, y su tiempo alcanza para oír cien grabaciones.

Ese equipo necesita tres respuestas:

1. **Predecir.** ¿Qué especie canta en esta grabación?
2. **Explicar.** ¿Qué características del sonido separan a las especies, y qué especies confunde el modelo entre sí?
3. **Decidir.** ¿Qué grabaciones revisa primero el experto, si solo alcanza a oír cien?

**El dato.** BirdCLEF+ 2025 es una competencia de reconocimiento de especies por su sonido, organizada alrededor de la Reserva Natural El Silencio, en el Magdalena Medio. Sus grabaciones de entrenamiento vienen de xeno-canto y de iNaturalist, donde las suben aficionados y científicos de muchos países. La competencia vive en Kaggle y exige cuenta. Aquí usamos una muestra pública de mil de esas grabaciones, 20 por cada una de 50 especies de aves, que se descarga de Hugging Face sin cuenta.

**La métrica.** Con 50 especies igual de frecuentes, quien adivina al azar acierta 1 de cada 50. Medimos la exactitud por grabación y la exactitud entre las tres primeras:

$$\text{exactitud}=\frac{1}{n}\sum_{i=1}^{n}\mathbb 1\{\hat y_i=y_i\},\qquad \text{exactitud}_{\text{top-3}}=\frac{1}{n}\sum_{i=1}^{n}\mathbb 1\{y_i\in\{\hat y_i^{(1)},\hat y_i^{(2)},\hat y_i^{(3)}\}\}.$$

La segunda importa porque al experto le sirve una lista corta de candidatas.

### Predicción escrita

- Con 12 grabaciones de entrenamiento por especie, ¿qué exactitud alcanzará el mejor modelo? **Mi apuesta:**
- Si corto las grabaciones en ventanas de cinco segundos y separo entrenamiento y prueba barajando ventanas, ¿la exactitud sube, baja o queda igual? **Mi apuesta:**
- Si el experto oye las cien grabaciones en que el modelo está menos seguro, ¿qué fracción de los errores del modelo alcanza a corregir? **Mi apuesta:**

## 2. Datos

La tabla trae una fila por grabación: especie, colección de origen, autor, licencia, coordenadas y duración. El sonido se pide aparte con `grabaciones(tabla)`, que entrega las grabaciones de a una como `(fila, onda, fs)`: la fila de la tabla, la onda como arreglo de numpy en una sola pista y la frecuencia de muestreo.

In [ ]:
aves = descargar("aves_magdalena")
print("Grabaciones:", len(aves), "| Especies:", aves["primary_label"].nunique(), "| Horas de sonido:", round(aves["segundos"].sum() / 3600, 1))
aves.head(3)

### 2.1 Conocer el dato antes de modelarlo

Respondan con código y escriban la lectura debajo de cada celda.

In [ ]:
# Esta celda debe explorar la tabla `aves` y mostrar, con números y tres gráficas:
#   - cuántas grabaciones hay por especie (primary_label) y de qué colección vienen (collection);
#   - qué licencias tienen (license) y cuántos autores distintos hay (author);
#   - la distribución de la duración (segundos) y de la calificación de calidad (rating);
#   - dónde se grabaron (latitude, longitude) y cuántas caen dentro de un rectángulo
#     aproximado alrededor de Colombia: latitud entre -4,3 y 13,5, longitud entre -79,1 y -66,8.

**Preguntas de lectura.**

- ¿Cuántas de las mil grabaciones se hicieron en Colombia o cerca? ¿Qué le dice eso sobre un modelo que se va a usar en el Magdalena Medio?
- La calificación 0 no significa «mala». ¿De qué colección vienen esas grabaciones? Compruébenlo con una tabla cruzada antes de usar `rating` para algo.
- Un mismo autor aporta varias grabaciones. ¿Por qué eso puede inflar el resultado, igual que el paciente repetido de la demostración?

**Mi lectura:**

In [ ]:
# Esta celda debe tomar UNA grabación de buena calidad (rating de 4,5 o más, entre 8 y 30 segundos),
# usando `next(grabaciones(tabla_de_una_fila))`, que devuelve (fila, onda, fs). Debe:
#   - imprimir la especie, el autor, la licencia y la frecuencia de muestreo fs;
#   - dejar oír los primeros 5 segundos con IPython.display.Audio;
#   - dibujar la onda en el tiempo y, debajo, su espectrograma en escala mel
#     (librosa.feature.melspectrogram con n_fft=1024, hop_length=512, n_mels=48, fmin=300, fmax=14000), en decibeles.

### 2.2 De la onda a la tabla

Una grabación de 30 segundos a 32.000 Hz son 960.000 números. Para llevarla a una tabla seguimos el mismo camino de la demostración, con las herramientas del audio.

**Ventanas.** Cada grabación se corta en ventanas de 5 segundos, sin traslape, hasta completar `SEGUNDOS_OIDOS`. Las grabaciones más cortas que una ventana se completan con silencio. Cada ventana hereda la especie de su grabación.

**Espectrograma.** La transformada de Fourier de tiempo corto mira la señal por tramos de $N$ muestras con una ventana $w$ que se desliza de a $H$ muestras:

$$X[m,k]=\sum_{n=0}^{N-1}x[n+mH]\,w[n]\,e^{-2\pi i\,kn/N}.$$

**Escala mel.** El oído distingue mejor las frecuencias bajas que las altas. La escala mel imita esa percepción, $\text{mel}(f)=2595\,\log_{10}(1+f/700)$, y la energía se suma en $M=48$ bandas repartidas de manera uniforme en mel entre 300 y 14.000 Hz. Sea $L[b,m]$ la energía de la banda $b$ en el tramo $m$, en decibeles.

**Piso de ruido.** El río, el viento y el micrófono ponen un ruido de fondo distinto en cada grabación. A cada banda se le resta su mediana en el tiempo, $\tilde L[b,m]=L[b,m]-\operatorname{mediana}_m L[b,m]$, para que quede lo que sobresale del fondo.

**MFCC.** Los coeficientes cepstrales resumen la forma del espectro de cada tramo con una transformada de coseno:

$$c_k[m]=\sum_{b=0}^{M-1}\tilde L[b,m]\,\cos\!\Bigl[\frac{\pi k}{M}\bigl(b+\tfrac12\bigr)\Bigr],\qquad k=0,\dots,19.$$

De cada ventana salen dos familias de columnas. La familia `mfcc_` trae la media y la desviación de cada coeficiente en el tiempo, y la desviación de su cambio entre tramos vecinos. La familia `mel_` trae, por banda, la desviación, el percentil 90 y el máximo.

In [ ]:
# Esta celda debe convertir las mil grabaciones en una tabla con una fila por VENTANA de 5 segundos.
#   - Recorrer `for fila, onda, fs in grabaciones(aves):` (no cargar todo en memoria a la vez).
#   - Cortar cada onda en ventanas de 5 s sin traslape, máximo SEGUNDOS_OIDOS // 5 ventanas por grabación,
#     desde el comienzo. Si la grabación dura menos de 5 s, completarla con ceros.
#   - Para cada ventana: espectrograma mel (n_fft=1024, hop_length=512, n_mels=48, fmin=300, fmax=14000),
#     pasarlo a decibeles, restarle a cada banda su mediana en el tiempo (piso de ruido),
#     sacar 20 MFCC con librosa.feature.mfcc(S=..., n_mfcc=20) y resumir:
#       media, desviación y desviación del cambio de cada MFCC (60 columnas `mfcc_...`);
#       desviación, percentil 90 y máximo de cada banda mel (144 columnas `mel_...`).
#   - Dejar: `C` (DataFrame de características) y `de_grabacion` (arreglo con el `indice` de la
#     grabación de cada ventana). Imprimir cuántas ventanas salieron, el tiempo y la frecuencia de muestreo.

### 2.3 PCA: ¿se ven las especies en la tabla?

Estandaricen las columnas y proyecten las ventanas sobre los dos primeros componentes principales. Si $Z$ es la tabla estandarizada, el PCA descompone $R=\tfrac{1}{n-1}Z^\top Z=V\Lambda V^\top$ y la fracción de varianza del componente $j$ es $\lambda_j/\sum_k\lambda_k$.

**Predicción escrita:** con 204 columnas, ¿cuánta varianza explicarán los dos primeros componentes? **Mi apuesta:**

In [ ]:
# Esta celda debe estandarizar `C`, ajustar un PCA de 10 componentes y dibujar:
#   - a la izquierda, la varianza explicada por componente;
#   - a la derecha, las ventanas en CP1 contra CP2, en gris, y resaltadas en color las de cuatro especies.
# Debe imprimir la varianza que explican los dos primeros componentes. Es un análisis descriptivo:
# este PCA no entra al modelo.

**Mi lectura** (¿las cuatro especies forman grupos separados o se traslapan?):

## 3. Modelado

### 3.1 El protocolo y la línea base

Las grabaciones se reparten en tres grupos, estratificando por especie: 60 % para entrenar, 20 % para validar y 20 % para probar. Eso deja 12, 4 y 4 grabaciones por especie. **Se reparten grabaciones, no ventanas**: todas las ventanas de una grabación quedan del mismo lado.

El modelo predice por ventana y la grabación se decide promediando. Si la grabación $i$ tiene ventanas $v=1,\dots,n_i$ y el modelo da $\hat p_{v}(k)$ para la especie $k$,

$$\bar p_i(k)=\frac{1}{n_i}\sum_{v=1}^{n_i}\hat p_{v}(k),\qquad \hat y_i=\arg\max_k\ \bar p_i(k).$$

Hay dos líneas base. La primera es adivinar: 1 de cada 50, o 2 %. La segunda es una regresión logística con una sola familia pobre, las tres primeras columnas de MFCC.

In [ ]:
# Esta celda debe:
#   - codificar la especie (primary_label) como entero: `y_grab` por grabación y `y_vent` por ventana;
#   - repartir las GRABACIONES (aves["indice"]) en 60 % / 20 % / 20 %, estratificando por especie, con random_state=0,
#     y construir las máscaras de ventanas `E`, `V`, `P_` con np.isin(de_grabacion, ...);
#   - comprobar con un `assert` que ninguna grabación quedó en dos grupos;
#   - definir `por_grabacion(probabilidades, tramo, grabs)`, que promedia las probabilidades de las ventanas de cada grabación,
#     y `exactitudes(modelo, columnas, tramo, grabs)`, que devuelve la exactitud top-1 y top-3 POR GRABACIÓN;
#   - calcular las dos líneas base: el azar (1/50) y una regresión logística con solo
#     las columnas mfcc_media_0, mfcc_media_1 y mfcc_media_2.

### 3.2 Cinco modelos

| Modelo | Idea en una línea |
|---|---|
| Regresión logística | $P(y=k\mid x)=e^{\beta_k^\top x}/\sum_j e^{\beta_j^\top x}$ |
| Análisis discriminante lineal | Supone una normal por especie con la misma covarianza $\Sigma$: $\delta_k(x)=x^\top\Sigma^{-1}\mu_k-\tfrac12\mu_k^\top\Sigma^{-1}\mu_k+\log\pi_k$ |
| $k$ vecinos | Vota con las ventanas de entrenamiento más parecidas |
| Bosque aleatorio | Promedia muchos árboles entrenados sobre remuestreos |
| Red neuronal pequeña | Una capa oculta de 128 unidades |

La regla: gana el de mayor exactitud por grabación en **validación**, y su exactitud en **prueba** es la que se reporta.

In [ ]:
# Esta celda debe definir `cinco_modelos()` (regresión logística, análisis discriminante lineal con encogimiento,
# k vecinos, bosque aleatorio y una red neuronal de una capa oculta; los que usan distancias o gradientes
# van dentro de un pipeline con StandardScaler), entrenarlos con las ventanas de entrenamiento y medir
# con `exactitudes` la exactitud POR GRABACIÓN en validación y en prueba, y la top-3 en prueba.
# Debe mostrar una tabla con las líneas base y los cinco modelos, una gráfica de barras, y dejar:
#   `entrenados` (diccionario de modelos ya ajustados), `comparacion` (la tabla) y `mejor`
#   (el nombre del ganador EN VALIDACIÓN).

Comparen con su primera apuesta. **Mi lectura** (¿cuántas veces mejor que el azar?, ¿sirve para algo una exactitud así?):

### 3.3 La trampa: barajar ventanas

Repitan el experimento de la demostración. Tomen dos de los modelos y sepárenlos de dos maneras: barajando ventanas con `train_test_split`, y por grabación como en 3.1. Midan la exactitud **por ventana** en ambos casos para que la comparación sea pareja.

In [ ]:
# Esta celda debe mostrar la trampa con números. Para k vecinos y para el bosque:
#   - separar las VENTANAS al azar (train_test_split, 20 % de prueba, estratificado, random_state=0),
#     entrenar y medir la exactitud por ventana;
#   - medir la exactitud por ventana del mismo modelo con la separación por grabación de 3.1.
# Debe mostrar las dos exactitudes lado a lado, la diferencia, y contar cuántas grabaciones de la prueba
# barajada tienen otra ventana suya en el entrenamiento.

Comparen con su segunda apuesta. **Mi explicación** (¿qué aprendió el modelo cuando se barajaron las ventanas?, ¿cuál de las dos cifras le pasará al equipo de la reserva con una grabación nueva?):

### 3.4 Explicar

Dos preguntas. ¿Qué familia de columnas sostiene el resultado? ¿Qué especies confunde el modelo entre sí?

In [ ]:
# Esta celda debe responder las dos preguntas con el modelo ganador, sobre la VALIDACIÓN:
#   - entrenarlo con solo las columnas `mfcc_`, con solo las `mel_` y con todas, y comparar la exactitud por grabación;
#   - construir la matriz de confusión por grabación (50 x 50), dibujarla, e imprimir las seis
#     confusiones más frecuentes con el nombre común de las especies, las especies que acierta siempre
#     y cuántas no acierta nunca.

**Mi lectura** (¿las especies que se confunden tienen algo en común?, ¿qué le preguntarían al ornitólogo?):

## 4. Comunicación

### 4.1 Decidir: las cien grabaciones del experto

El modelo se equivoca en muchas grabaciones y el experto solo puede oír `REVISIONES`. La propuesta es mandarle primero aquellas en que el modelo está menos seguro. La confianza de la grabación $i$ es la probabilidad promedio de la especie ganadora,

$$c_i=\max_k\ \bar p_i(k),$$

y el experto recibe las `REVISIONES` grabaciones de prueba con menor $c_i$. Supongan que el experto siempre acierta. Comparen tres políticas sobre la prueba: que el experto no oiga nada, que oiga cien escogidas al azar y que oiga las cien de menor confianza.

La política al azar tiene una exactitud esperada que se calcula sin simular. Si el modelo tiene exactitud $a$ y el experto revisa una fracción $r$ de las grabaciones, el sistema acierta $a+r\,(1-a)$.

In [ ]:
# Esta celda debe, con el modelo ganador y SOLO sobre la prueba:
#   - calcular por grabación la especie predicha y la confianza (la mayor probabilidad promedio);
#   - ordenar las grabaciones de menor a mayor confianza y marcar las primeras REVISIONES para el experto;
#   - comparar tres exactitudes del sistema completo, suponiendo que el experto no se equivoca:
#     modelo solo; experto oye REVISIONES al azar (fórmula a + r(1 - a), y además el rango de 2.000 sorteos);
#     experto oye las REVISIONES de menor confianza;
#   - dibujar la exactitud del modelo sobre las grabaciones más seguras a medida que se le dejan más;
#   - mostrar las diez primeras grabaciones de la lista del experto, con archivo, especie y url.

Comparen con su tercera apuesta. Miren si la política de menor confianza cae dentro o fuera del rango de los sorteos al azar. Esa comparación dice si la confianza del modelo sirve para ordenar el trabajo del experto. **Mi lectura:**

### La decisión en tres frases

Para la directora de la reserva, que no sabe qué es un MFCC.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Auditoría del código que escribió la IA

Antes de creerle a sus números, comprueben tres cosas en el código que pegaron. Escriban qué encontraron y, si algo estaba mal, cómo lo corrigieron.

1. **¿La partición es por grabación?** Busquen dónde se separan entrenamiento, validación y prueba. Comprueben con código que ninguna grabación tiene ventanas en dos grupos. Una IA tiende a escribir `train_test_split(C, y)` sobre las ventanas, que es la trampa de 3.3. **Lo que encontré:**
2. **¿Se respetó la frecuencia de muestreo y se oyó lo que se dijo?** Si la IA usó `librosa.load`, revisen si puso `sr=None`: por defecto esa función remuestrea a 22.050 Hz y corta todo lo que suena por encima de 11.025 Hz. Comprueben también que cada ventana tiene `5 * fs` muestras y que no se usaron más de `SEGUNDOS_OIDOS` por grabación. **Lo que encontré:**
3. **¿Algo se ajustó con datos de validación o de prueba?** Revisen que el `StandardScaler` esté dentro del pipeline de cada modelo, que el ganador se haya escogido con la columna de validación y que la exactitud reportada sea por grabación y no por ventana. **Lo que encontré:**

### 4.3 Bitácora de decisiones

Llenen las filas vacías.

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Muestra de 1.000 grabaciones de Hugging Face | Las grabaciones completas de Kaggle | No exige cuenta y pesa 257 MB |
| Ventanas de 5 segundos desde el comienzo | | |
| Restar el piso de ruido por banda | | |
| Partición por grabación, 60 / 20 / 20 | | |
| Promediar probabilidades de las ventanas | | |
| Escoger el modelo en validación | | |
| Mandar al experto las de menor confianza | | |

### 4.4 Riesgos y mantenimiento

- Las grabaciones de entrenamiento son de alguien que apuntó el micrófono a un ave. Las de una grabadora fija en el bosque traen varias especies a la vez, lejos y con lluvia. ¿Qué le pasará a la exactitud?
- Buena parte de las grabaciones se hicieron fuera de Colombia. ¿Por qué puede importar, si la especie es la misma?
- Cada grabación tiene autor y licencia, casi todas de uso no comercial. ¿Qué debe hacer el equipo antes de publicar un producto con este modelo, y cómo se le da crédito a quien grabó?
- Publicar dónde canta una especie amenazada puede ponerla en riesgo. ¿Qué parte de los resultados no pondrían en un mapa abierto?
- El modelo solo conoce 50 especies. ¿Qué responde cuando canta la número 51, y cómo lo detectarían?

### 4.5 Cambien una línea

Vuelvan a la celda de parámetros, cambien `SEGUNDOS_OIDOS = 30` por `SEGUNDOS_OIDOS = 5` y corran todo de nuevo. Con 5 segundos el modelo oye una sola ventana por grabación.

| | 30 segundos | 5 segundos |
|---|---|---|
| Ventanas en total | | |
| Ganador en validación | | |
| Exactitud del ganador en prueba | | |
| Exactitud entre las tres primeras, en prueba | | |
| Exactitud del sistema con el experto oyendo las de menor confianza | | |

Con la tabla llena, respondan en tres líneas: ¿cuánto se gana por oír más de cada grabación? ¿De dónde creen que viene la ganancia, de tener más ventanas para entrenar o de promediar más ventanas al predecir? **Mi respuesta:**

### 4.6 Procedencia y entrega

In [ ]:
{nombre: ficha for nombre, ficha in procedencia().items() if nombre == "aves_magdalena"}

**Para entregar:** en Colab, menú *Archivo*, *Guardar una copia en GitHub*, y escojan su repositorio de portafolio, carpeta `clase08/`. Antes de guardar usen *Entorno de ejecución*, *Reiniciar sesión y ejecutar todo*: un cuaderno que no corre de principio a fin desde cero no se califica.